# Custo de buscas e índices

Procure o último ID e um ausente em uma lista. Antes da busca, estime o número de comparações. Retome [percorrer sequências](https://devrgs.github.io/Aprendendo-linguagens/#/python/sequencias/percorrer-e-filtrar) e [conjuntos](https://devrgs.github.io/Aprendendo-linguagens/#/python/registros/conjuntos).

Uma busca linear compara elementos até achar o alvo ou acabar a lista. Seu pior caso cresce na proporção do comprimento. Um `set` cria um índice de pertinência por hash, com busca usual de custo médio constante sob condições típicas. Sua construção percorre os dados, guarda as chaves únicas e usa memória adicional. O custo de colisões depende dos dados, portanto o exemplo conta passagens e consultas, e deixa a medição de tempo para uma comparação controlada.

No caderno e no kit, rode os mesmos valores. Em E2, compare contagens, não segundos. Quantas consultas justificam construir um índice neste cenário? Em seguida, veja [reprodutibilidade](https://devrgs.github.io/Aprendendo-linguagens/#/python/qualidade/reprodutibilidade).

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-custo-de-operacoes-', dir=_raiz))


## E1 · Preveja e execute

Conte comparações ao buscar 40 e 99 em [10, 20, 30, 40]. Identifique se uma busca para antes da outra.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
def buscar_lista(valores, alvo):
    comparacoes = 0
    for valor in valores:
        comparacoes += 1
        if valor == alvo:
            return True, comparacoes
    return False, comparacoes

ids = [10, 20, 30, 40]
for alvo in [40, 99]:
    print(buscar_lista(ids, alvo))


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
(True, 4)
(False, 4)
```

40 está na última posição. 99 não existe. Ambos os casos examinam todos os itens, embora a resposta de pertinência seja diferente.


## E2 · Altere ou depure

Para quatro IDs e três consultas, compare a busca repetida na lista com a construção de um único set e consultas de pertinência. Confira as mesmas respostas.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
def buscar_lista(valores, alvo):
    comparacoes = 0
    for valor in valores:
        comparacoes += 1
        if valor == alvo:
            return True, comparacoes
    return False, comparacoes

ids = [10, 20, 30, 40]
consultas = [40, 99, 10]

resultados = [buscar_lista(ids, alvo) for alvo in consultas]
indice = set(ids)
respostas = [alvo in indice for alvo in consultas]
print([encontrado for encontrado, _ in resultados] == respostas)
print("comparações na lista:", sum(contagem for _, contagem in resultados))
print("itens lidos para construir:", len(ids))
print("consultas ao índice:", len(consultas))


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
True
comparações na lista: 9
itens lidos para construir: 4
consultas ao índice: 3
```

As comparações lineares somam 4 + 4 + 1. O índice usa uma passagem inicial e três operações de pertinência. Estas são contagens de atividades diferentes, pois cada consulta ao set pode ter mais de uma operação interna.


## E3 · Resolva

Confirme o acordo entre lista e índice para 100 IDs e 1000 consultas alternadas ao último e a um ausente. Conte comparações da lista, itens da construção e consultas ao índice. Confira a pertinência em entrada vazia e justifique quando reaproveitar o índice.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
def buscar_lista(valores, alvo):
    comparacoes = 0
    for valor in valores:
        comparacoes += 1
        if valor == alvo:
            return True, comparacoes
    return False, comparacoes

ids = list(range(100))
consultas = [99, -1] * 500
indice = set(ids)
resultados = [buscar_lista(ids, alvo) for alvo in consultas]
iguais = [a for a, _ in resultados] == [alvo in indice for alvo in consultas]
print("respostas iguais:", iguais)
print("comparações:", sum(qtd for _, qtd in resultados))
print("construção e consultas:", len(ids), len(consultas))
print("vazio:", buscar_lista([], 99)[0] == (99 in set()))


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
respostas iguais: True
comparações: 100000
construção e consultas: 100 1000
vazio: True
```

São 100 comparações em cada busca de último ou ausente. Construir o índice uma vez favorece muitas consultas repetidas e custa memória para as chaves. Para uma consulta simples em uma lista pequena, a passagem de construção pode não compensar. Uma entrada vazia dá ausência nos dois casos.
